# 💬 Gynecological Review Analysis & Similarity Search (English Only)

This notebook processes patient reviews in **English** using Natural Language Processing (NLP):
1. **Load English Reviews**: Dataset of gynecological patient reviews (`gynecological_reviews.csv`).
2. **Text Preprocessing**: Cleaning, lowercasing, tokenization, English stopword removal, and Porter stemming using NLTK.
3. **TF-IDF Vectorization & Cosine Similarity**: Converts reviews into numerical vectors and finds the most similar review for any input query in English.

In [1]:
# Import libraries
import re
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download required NLTK datasets if not already present
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

stop_words = set(stopwords.words("english"))

In [2]:
# Load the reviews dataset
df = pd.read_csv("gynecological_reviews.csv")
df.head()

,Review
0,I have been diagnosed with UTI. My doctor give...
1,My doctor diagnosed me with a yeast infection....
2,I was diagnosed with bacterial vaginosis and m...
3,I have been experiencing pelvic pain and my gy...
4,"After my recent check-up, my doctor diagnosed ..."


In [4]:
df.tail()

,Review
82,I’ve been diagnosed with bacterial vaginosis a...
83,"After being diagnosed with a yeast infection, ..."
84,"Diagnosed with a bladder infection, I’m on a 7..."
85,I was recently diagnosed with cystitis and giv...
86,"Following my diagnosis of a UTI, I’m on a 7-da..."


In [5]:
print("Dataset Shape:", df.shape)

Dataset Shape: (87, 1)


## Text Preprocessing and Cosine Similarity (English Only)

In [6]:
def preprocess_text(text):
    """
    Preprocesses the input English text by:
    - Removing non-alphabetic characters
    - Converting to lowercase
    - Tokenizing into words
    - Removing English stopwords
    - Applying Porter Stemmer
    
    Parameters:
    text (str): The input text to preprocess.
    
    Returns:
    str: The cleaned and stemmed text.
    """
    ps = PorterStemmer()
    current_stop = set(stopwords.words("english"))
    text = re.sub("[^a-zA-Z]", " ", str(text)).lower()
    words = word_tokenize(text)
    return " ".join(ps.stem(w) for w in words if w not in current_stop)

def find_most_similar_review(input_text, df):
    """
    Finds the review most similar to the input English text based on cosine similarity.
    
    Parameters:
    input_text (str): The input text to compare with the reviews.
    df (pd.DataFrame): DataFrame containing a "Review" column with English reviews.
    
    Returns:
    str: The most similar review from the dataframe.
    """
    input_text_processed = preprocess_text(input_text)
    df["processed_review"] = df["Review"].apply(preprocess_text)
    reviews = df["processed_review"].tolist() + [input_text_processed]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(reviews)
    cosine_sim = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])
    most_similar_index = cosine_sim.argmax()
    return df["Review"].iloc[most_similar_index]

In [7]:
# Test with an English query
input_text = "I was diagnosed with a urinary tract infection and have pain while urinating. What should I do?"
most_similar = find_most_similar_review(input_text, df)
print("Input Query (English):", input_text)
print("\nMost Similar Matched Review (English):\n", most_similar)

Input Query (English): I was diagnosed with a urinary tract infection and have pain while urinating. What should I do?

Most Similar Matched Review (English):
 I was recently diagnosed with a urinary tract infection and am on antibiotics for a week. Is it normal for my urine to smell odd and for me to feel a burning sensation? I’ve been instructed to drink more water and take it easy.


In [8]:
# Additional test queries in English
test_queries = [
    "My periods have been irregular and I am noticing facial hair growth and weight gain.",
    "I have severe lower abdominal discomfort and burning sensation when urinating.",
    "My doctor prescribed antibiotics for an infection and told me to drink plenty of fluids."
]

for i, query in enumerate(test_queries, 1):
    match = find_most_similar_review(query, df)
    print(f"--- Test Case {i} ---")
    print(f"Query: {query}")
    print(f"Matched: {match}\n")

--- Test Case 1 ---
Query: My periods have been irregular and I am noticing facial hair growth and weight gain.
Matched: My doctor identified polycystic ovary syndrome (PCOS) as the cause of my irregular periods. She prescribed oral contraceptives for regulation. Is it common to experience weight gain and acne with PCOS? The doctor also advised a balanced diet and regular exercise.

--- Test Case 2 ---
Query: I have severe lower abdominal discomfort and burning sensation when urinating.
Matched: My gynecologist diagnosed me with hydrosalpinx and suggested surgery to remove the affected fallopian tube. Is it normal to have lower abdominal pain and infertility with this condition? The doctor recommended discussing fertility options.

--- Test Case 3 ---
Query: My doctor prescribed antibiotics for an infection and told me to drink plenty of fluids.
Matched: y doctor diagnosed me with a UTI and prescribed antibiotics for a week. Is it usual to experience a strange smell in the urine and bu